# Katherine's Notebook
# P3: Text Exploration

REQ: Open notebooks with a standard header including a good title, your company/name/alias, a link to the repo, purpose, and date.

- Author: [Katherine McGaughey](https://github.com/k363m611/)
- Repository: [nlp-03-text-exploration](https://github.com/k363m611/nlp-03-text-exploration/)
- Date: 2026-03

Purpose

  Perform exploratory analysis of a small, controlled text corpus.
  Demonstrate how structure emerges from token distributions,
  category comparisons, and co-occurrence patterns.

Analytical Questions

- What tokens dominate each category?
- How do categories differ in vocabulary?
- What words appear in similar contexts?
- What structure is visible before using any models?

Notes

- This module focuses on exploratory analysis (EDA), not modeling.
- Results here prepare for later work with pipelines and embeddings.

Project Instructions

- See [docs/project-instructions.md](../docs/project-instructions.md)
- See [docs/glossary.md](../docs/glossary.md)

## Terminology

In preparation for large language models and related methods, our analysis does not begin with semantic interpretation.
Instead, we focus on **proximity** and observable **patterns** in the text.

We evaluate **co-occurrence (context windows)**, 
that is, _which words tend to appear near each other_.

The full collection of text is called a **corpus** (a set of documents).
For this analysis, each document is represented as a single line of text.

See the README.md for more


## Section 0. Intro to Jupyter Notebooks

This notebook explores a small text corpus and demonstrates tokenization,
frequency analysis, co-occurrence, and bigrams.

For Phase 4, I modified the tokenizer by adding stopword filtering.
This helps remove very common words so the analysis focuses more on
meaningful terms within each category.

Tips:
- Run a cell with Ctrl+Enter (Cmd+Enter on Mac)
- Select a kernel (your .venv)
- Use Run All before committing


## Section 1. Setup and Imports

Imports and configuration appear once, at the top.


In [ ]:
# Section 1 Python cell

from collections import defaultdict
import logging
from pathlib import Path

from datafun_toolkit.logger import get_logger, log_header, log_path
import matplotlib.pyplot as plt
import polars as pl

print("Imports complete.")

### Configure Logger and Paths


In [ ]:
# Section 1 Python cell (logger configuration and path setup)

LOG: logging.Logger = get_logger("CI", level="DEBUG")

NOTEBOOKS_PATH: Path = Path.cwd()
ROOT_PATH: Path = NOTEBOOKS_PATH.parent
SCRIPTS_PATH: Path = ROOT_PATH / "scripts"

log_header(LOG, "MODULE 3 NOTEBOOK: CORPUS EXPLORATION")

log_path(LOG, "ROOT_PATH", ROOT_PATH)
log_path(LOG, "NOTEBOOKS_PATH", NOTEBOOKS_PATH)
log_path(LOG, "SCRIPTS_PATH", SCRIPTS_PATH)

LOG.info("Logger configured.")

## Section 2. Define Corpus (Labeled Text Documents)


In [ ]:
# Section 2 Python cell

corpus = [
    {"category": "dog", "text": "A dog barks loudly."},
    {"category": "dog", "text": "The puppy runs in the yard."},
    {"category": "dog", "text": "A canine wears a leash."},
    {"category": "dog", "text": "The dog ran across the yard."},
    {"category": "cat", "text": "A cat sleeps quietly."},
    {"category": "cat", "text": "The kitten plays with yarn."},
    {"category": "cat", "text": "A feline purrs softly."},
    {"category": "cat", "text": "The cat slept near the window."},
    {"category": "car", "text": "A car drives on the road."},
    {"category": "car", "text": "The sedan parks in the garage."},
    {"category": "car", "text": "A vehicle has four wheels."},
    {"category": "truck", "text": "A truck carries cargo."},
    {"category": "truck", "text": "The pickup pulls a trailer."},
    {"category": "truck", "text": "The truck hauls heavy loads."},
]

print(f"Corpus contains {len(corpus)} documents.")

## Section 3. Tokenize and Clean Text


In [ ]:
# Section 3 Python cell

# Tokenization splits text into word-like units.

STOPWORDS = {"the", "and", "for", "with", "a", "an", "in", "on", "has"}

def tokenize(text: str) -> list[str]:
    tokens = text.lower().split()
    cleaned = [t.strip(".,:;!?()[]\"'") for t in tokens]
    return [t for t in cleaned if len(t) > 2 and t not in STOPWORDS]


records_list: list[dict[str, str]] = []

for doc in corpus:
    tokens = tokenize(doc["text"])
    for token in tokens:
        records_list.append({"category": doc["category"], "token": token})

token_df: pl.DataFrame = pl.DataFrame(records_list)

print("Tokenization complete.")
print(token_df.head(10))

## Section 4. Compute Global Token Frequencies


In [ ]:
# Section 4 Python cell

global_freq_df: pl.DataFrame = (
    token_df.group_by("token").len().sort("len", descending=True)
)

print("Top global tokens:")
print(global_freq_df.head(10))

## Section 5. Compute Token Frequencies by Category


In [ ]:
# Section 5 Python cell

category_freq_df: pl.DataFrame = (
    token_df.group_by(["category", "token"])
    .len()
    .sort(["category", "len"], descending=True)
)

print("Top tokens by category:")
print(category_freq_df.head(12))

## Section 6. Identify Top Tokens per Category


In [ ]:
# Section 6 Python cell

top_per_category_dict: dict[str, list[str]] = {}

for category in token_df["category"].unique().to_list():
    subset_df = category_freq_df.filter(pl.col("category") == category).head(5)
    top_tokens_list = subset_df["token"].to_list()
    top_per_category_dict[category] = top_tokens_list
    print(f"{category.upper()} top tokens: {top_tokens_list}")

## Section 7. Analyze Co-occurrence (Context Windows)


In [ ]:
# Section 7 Python cell

WINDOW_SIZE: int = 2

co_occurrence_dict: dict[str, list[str]] = defaultdict(list)

for doc in corpus:
    tokens = tokenize(doc["text"])
    for i, token in enumerate(tokens):
        start = max(0, i - WINDOW_SIZE)
        end = min(len(tokens), i + WINDOW_SIZE + 1)
        context = tokens[start:end]
        for ctx in context:
            if ctx != token:
                co_occurrence_dict[token].append(ctx)

for target in ["dog", "cat", "car", "truck"]:
    print(f"\nContext for '{target}':")
    print(co_occurrence_dict[target][:10])

## Section 8. Create Bigrams (Local Word Pairs) and Compute Frequencies


In [ ]:
# Section 8 Python cell

bigrams_list: list[tuple[str, str]] = []

for doc in corpus:
    tokens = tokenize(doc["text"])
    for i in range(len(tokens) - 1):
        bigrams_list.append((tokens[i], tokens[i + 1]))

bigram_df: pl.DataFrame = pl.DataFrame(
    {"bigram": [f"{a} {b}" for a, b in bigrams_list]}
)

bigram_freq_df: pl.DataFrame = (
    bigram_df.group_by("bigram").len().sort("len", descending=True)
)

print("Top bigrams:")
print(bigram_freq_df.head(10))

## Section 9. Visualize Token Frequencies


In [ ]:
# Section 9 Python cell

selected_category = "dog"
selected_df = category_freq_df.filter(pl.col("category") == selected_category).head(5)

plt.figure(figsize=(8, 4))
plt.bar(selected_df["token"], selected_df["len"])

ax = plt.gca()
ax.tick_params(axis="x", labelrotation=45)

plt.title(f"Top Tokens ({selected_category.title()} Category)")
plt.xlabel("Token")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

## Section 10. Interpret Results and Identify Patterns


In [ ]:
# Section 10 Python cell

print("\nCASE GENERAL OBSERVATIONS:")

print("- Tokens cluster by category (dog, cat, car, truck).")
print("- Words that appear in similar contexts behave similarly.")
print("- Co-occurrence reveals contextual relationships between words.")
print("- Bigrams capture local structure beyond single tokens.")
print("- Patterns emerge before any machine learning is applied.")

print("\nKATHERINE SPECIFIC OBSERVATIONS:")
print("- After stopword filtering, the token lists became cleaner and easier to interpret.")
print("- Category-specific words stood out more clearly after removing common filler words.")
print("- The frequency and bigram results focused more on meaningful patterns in each category.")
print("- Small preprocessing changes can noticeably improve exploratory text analysis.")

In [ ]:
# Final Python cell

LOG.info("========================")
LOG.info("Notebook executed successfully!")
LOG.info("========================")

print("Notebook executed successfully.")